# CTF Solver — Kaggle/Colab Backend

Serve LLM trên Kaggle (2x T4 = 32GB free) hoặc Colab (T4/A100) và expose qua ngrok.

**Kaggle**: Settings → Accelerator → GPU T4 x2  
**Colab**: Runtime → Change runtime type → GPU

Chạy từng cell theo thứ tự từ trên xuống.


In [ ]:
# Cell 1 — Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# Cell 2 — Cài dependencies
!pip install vllm pyngrok -q

import torch, subprocess, sys, os, shutil

cuda_maj = torch.version.cuda.split('.')[0]  # "12"
cuda_min = torch.version.cuda.split('.')[1]  # "4"
cuda_tag = f"cu{cuda_maj}{cuda_min}"         # "cu124"
torch_tag = '.'.join(torch.__version__.split('.')[:2])  # "2.5"
whl_url = f"https://flashinfer.ai/whl/{cuda_tag}/torch{torch_tag}/"
print(f"CUDA: {torch.version.cuda}, PyTorch: {torch.__version__}")
print(f"Trying pre-built flashinfer from {whl_url}")

# Cài pre-built flashinfer (có compiled kernels, không cần JIT)
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'flashinfer-python',
     '--find-links', whl_url, '--quiet'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"Pre-built wheel not found for {cuda_tag}/torch{torch_tag}, falling back to JIT.")
else:
    print("Pre-built flashinfer installed.")

# Xoá JIT cache cũ để tránh dùng artifacts hỏng
cache_dir = os.path.expanduser('~/.cache/flashinfer')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"Cleared flashinfer JIT cache: {cache_dir}")

# vLLM 0.19+ always uses V1 engine with FlashInfer — no toggle needed
# Cell 3b sets up the libcuda.so stub required for FlashInfer JIT
print('Done')

In [ ]:
# Cell 3 — Config
# ─── Chọn model dựa trên VRAM ────────────────────────────────────────────────
vram_list = !nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits
total_vram = sum(int(x.strip()) for x in vram_list) // 1024
print(f'Total VRAM: {total_vram}GB')

if total_vram >= 30:
    MODEL = 'Qwen/Qwen2.5-32B-Instruct-AWQ'   # Kaggle 2xT4 (32GB)
    QUANTIZATION = 'awq'
    MAX_LEN = 16384
elif total_vram >= 20:
    MODEL = 'Qwen/Qwen2.5-14B-Instruct-AWQ'   # Colab A100 40GB
    QUANTIZATION = 'awq'
    MAX_LEN = 16384
else:
    MODEL = 'Qwen/Qwen2.5-7B-Instruct-AWQ'    # Colab T4 15GB
    QUANTIZATION = 'awq'
    MAX_LEN = 8192

MODEL_NAME = MODEL.split('/')[-1]
print(f'Model: {MODEL}')

# ─── Ngrok token ──────────────────────────────────────────────────────────────
# Lấy tại: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ''  # ← dán token vào đây

In [ ]:
# Cell 3b — Tạo libcuda.so stub cho flashinfer JIT linker
#
# /usr/local/cuda/lib64/stubs/ là read-only trên Kaggle.
# Fix: tạo stub ở /usr/local/nvidia/lib64/ (cùng chỗ với libcuda.so.1)
# hoặc /tmp/cuda-stubs/ rồi set LIBRARY_PATH để linker tìm thấy.
#
# QUAN TRỌNG: KHÔNG xoá ~/.cache/flashinfer/ — cache này tồn tại qua các session
# Kaggle (home dir persistent). Xoá cache = phải compile lại >30 phút trên T4.

import os, shutil, subprocess

NVIDIA_LIB = '/usr/local/nvidia/lib64'
STUB_SRC = os.path.join(NVIDIA_LIB, 'libcuda.so.1')

stub_dir = None

# Thử 1: tạo libcuda.so trong /usr/local/nvidia/lib64/ (cùng dir với libcuda.so.1)
if os.path.exists(STUB_SRC):
    target = os.path.join(NVIDIA_LIB, 'libcuda.so')
    if os.path.exists(target) or os.path.islink(target):
        print(f'Already exists: {target}')
        stub_dir = NVIDIA_LIB
    else:
        try:
            os.symlink(STUB_SRC, target)
            print(f'Symlink: {target} -> {STUB_SRC}')
            stub_dir = NVIDIA_LIB
        except OSError as e:
            print(f'Cannot symlink in {NVIDIA_LIB}: {e}')

# Thử 2: /tmp/cuda-stubs/ (luôn writable) + gcc empty stub
if stub_dir is None:
    tmp_dir = '/tmp/cuda-stubs'
    os.makedirs(tmp_dir, exist_ok=True)
    target = os.path.join(tmp_dir, 'libcuda.so')

    for src_pat in [STUB_SRC, '/usr/lib/x86_64-linux-gnu/libcuda.so.1']:
        if os.path.exists(src_pat) and not (os.path.exists(target) or os.path.islink(target)):
            try:
                os.symlink(src_pat, target)
                print(f'Symlink in /tmp: {target} -> {src_pat}')
                stub_dir = tmp_dir
                break
            except OSError:
                pass

    if stub_dir is None:
        r = subprocess.run(
            ['gcc', '-shared', '-Wl,-soname,libcuda.so.1',
             '-Wl,--allow-shlib-undefined', '-o', target, '/dev/null'],
            capture_output=True, text=True
        )
        if r.returncode == 0:
            print(f'gcc stub: {target} ({os.path.getsize(target)} bytes)')
            stub_dir = tmp_dir
        else:
            print(f'gcc failed: {r.stderr.strip()}')

# Set LIBRARY_PATH — inherit vào tất cả subprocesses (Popen → vllm workers → ninja → ld)
if stub_dir:
    existing = os.environ.get('LIBRARY_PATH', '')
    paths = [stub_dir] + [p for p in existing.split(':') if p and p != stub_dir]
    os.environ['LIBRARY_PATH'] = ':'.join(paths)
    print(f'LIBRARY_PATH={os.environ["LIBRARY_PATH"]}')

# Kiểm tra cache flashinfer
cache_dir = os.path.expanduser('~/.cache/flashinfer')
if os.path.exists(cache_dir):
    cached = list(os.listdir(cache_dir))
    print(f'FlashInfer JIT cache exists ({len(cached)} entries) — keeping it (tránh recompile)')
else:
    print('FlashInfer JIT cache empty — lần đầu chạy sẽ compile ~30-60 phút trên T4')

print('Done.')

In [ ]:
# Cell 4 — Start vLLM (streaming log, tự dừng khi ready)
import subprocess, time, urllib.request, os

LOG_FILE = '/tmp/vllm.log'

# ── Giải phóng VRAM từ processes cũ ─────────────────────────────────────────
my_pid = os.getpid()
result = subprocess.run(
    ['nvidia-smi', '--query-compute-apps=pid,used_memory', '--format=csv,noheader'],
    capture_output=True, text=True
)
gpu_procs = result.stdout.strip()
print(f'GPU processes before start: {gpu_procs or "(none)"}')

killed = []
for line in gpu_procs.split('\n'):
    if not line.strip():
        continue
    pid_str = line.split(',')[0].strip()
    try:
        pid = int(pid_str)
        if pid != my_pid:
            os.kill(pid, 9)
            killed.append(pid)
    except (ValueError, ProcessLookupError):
        pass

if killed:
    print(f'Killed: {killed}')
    time.sleep(5)

vram_free = subprocess.run(
    ['nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'],
    capture_output=True, text=True
).stdout.strip().split('\n')
print(f'Free VRAM: {[v.strip() + " MiB" for v in vram_free]}')

# ── Start vLLM ───────────────────────────────────────────────────────────────
gpu_count_raw = !nvidia-smi --query-gpu=name --format=csv,noheader
GPU_COUNT = len([x for x in gpu_count_raw if x.strip()])
print(f'Detected {GPU_COUNT} GPU(s) → tensor-parallel-size={GPU_COUNT}')

cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL,
    '--port', '8000',
    '--host', '0.0.0.0',
    '--max-model-len', str(MAX_LEN),
    '--gpu-memory-utilization', '0.90',
    '--served-model-name', MODEL_NAME,
    '--trust-remote-code',
    '--quantization', QUANTIZATION,
    '--tensor-parallel-size', str(GPU_COUNT),
    '--no-enable-log-requests',
    '--enforce-eager',
    '--enable-auto-tool-choice',
    '--tool-call-parser', 'hermes',
]

# Force TRITON_ATTN backend — T4 (CC 7.5) có FA2 disabled, FlashInfer JIT mất >60 phút
# TRITON_ATTN compile trong vài giây, không cần JIT cache riêng
env = os.environ.copy()
env['VLLM_ATTENTION_BACKEND'] = 'TRITON_ATTN'
print('Forcing VLLM_ATTENTION_BACKEND=TRITON_ATTN (avoids FlashInfer JIT)')

log_f = open(LOG_FILE, 'w')
proc = subprocess.Popen(cmd, stdout=log_f, stderr=log_f, env=env)

print(f'vLLM starting PID={proc.pid}...')
print('Theo dõi log bên dưới:\n')

ready = False
start = time.time()
TIMEOUT = 1800  # 30 phút — TRITON_ATTN nhanh hơn nhiều
last_log_time = start

with open(LOG_FILE, 'r') as lf:
    while time.time() - start < TIMEOUT:
        line = lf.readline()
        if line:
            print(line, end='', flush=True)
            last_log_time = time.time()
            if 'Application startup complete' in line or 'Uvicorn running' in line:
                ready = True
                break
        else:
            try:
                urllib.request.urlopen('http://localhost:8000/health', timeout=2)
                ready = True
                break
            except:
                elapsed = int(time.time() - start)
                idle = int(time.time() - last_log_time)
                if idle > 0 and idle % 120 == 0:
                    print(f'[{elapsed//60}m elapsed] vLLM loading...', flush=True)
                time.sleep(2)

        if proc.poll() is not None:
            print(f'\nERROR: vLLM exit code={proc.returncode}')
            print('--- Last logs ---')
            !tail -50 {LOG_FILE}
            raise RuntimeError('vLLM crashed. Xem log ở trên.')

if ready:
    print(f'\n✓ vLLM ready sau {int(time.time() - start)}s')
else:
    print('\nTimeout. Xem log:')
    !tail -30 {LOG_FILE}

In [ ]:
# Cell 4b — Start LiteLLM proxy (Anthropic API → OpenAI → vLLM)
# ctf CLI dùng Anthropic SDK (POST /v1/messages), vLLM chỉ hiểu OpenAI format
# LiteLLM dịch giữa hai format này
#
# Dùng provider "hosted_vllm/" thay vì "openai/" để LiteLLM luôn gọi
# /v1/chat/completions thay vì /v1/responses (Responses API mới của OpenAI)
import subprocess, time, urllib.request, os

LITELLM_PORT = 4000
LITELLM_CONFIG = '/tmp/litellm_kaggle.yaml'

# Cài LiteLLM nếu chưa có
subprocess.run(['pip', 'install', 'litellm[proxy]', '-q'], check=True)

# Tạo config — hosted_vllm/ forces /v1/chat/completions (không phải /v1/responses)
with open(LITELLM_CONFIG, 'w') as f:
    f.write(f"""model_list:
  - model_name: {MODEL_NAME}
    litellm_params:
      model: hosted_vllm/{MODEL_NAME}
      api_base: http://localhost:8000/v1
      api_key: dummy

litellm_settings:
  drop_params: true
  request_timeout: 600
""")

print(f'LiteLLM config written to {LITELLM_CONFIG}')

# Kill any existing LiteLLM on this port
subprocess.run(['pkill', '-f', f'litellm.*{LITELLM_PORT}'], capture_output=True)
time.sleep(2)

# Start LiteLLM
log_f = open('/tmp/litellm.log', 'w')
proc = subprocess.Popen(
    ['litellm', '--config', LITELLM_CONFIG, '--port', str(LITELLM_PORT)],
    stdout=log_f, stderr=log_f
)
print(f'LiteLLM starting PID={proc.pid}...')

# Đợi ready
for i in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen(f'http://localhost:{LITELLM_PORT}/health', timeout=2)
        print(f'✓ LiteLLM ready on port {LITELLM_PORT}')
        break
    except:
        if proc.poll() is not None:
            print(f'ERROR: LiteLLM crashed')
            !tail -20 /tmp/litellm.log
            raise RuntimeError('LiteLLM crashed')
        if i % 5 == 4:
            print(f'  waiting... ({(i+1)*2}s)')
else:
    print('LiteLLM slow to start, check log:')
    !tail -10 /tmp/litellm.log

In [ ]:
# Cell 5 — Expose LiteLLM proxy qua ngrok
from pyngrok import ngrok, conf

if not NGROK_TOKEN:
    raise ValueError('Điền NGROK_TOKEN vào Cell 3!')

# Đóng tunnel cũ nếu có
ngrok.kill()

conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(4000, 'http')  # expose LiteLLM (Anthropic-compatible), not vLLM
PUBLIC_URL = tunnel.public_url

print('=' * 60)
print('  CTF Solver Backend READY')
print('=' * 60)
print(f'  URL   : {PUBLIC_URL}')
print(f'  Model : {MODEL_NAME}')
print()
print('  Copy và chạy trên máy local:')
print()
print(f'  export ANTHROPIC_BASE_URL={PUBLIC_URL}')
print( '  export ANTHROPIC_API_KEY=dummy')
print(f'  ctf --challenge ~/ctf/chall --model {MODEL_NAME}')
print('=' * 60)

In [ ]:
# Cell 5 — Expose qua ngrok
from pyngrok import ngrok, conf

if not NGROK_TOKEN:
    raise ValueError('Điền NGROK_TOKEN vào Cell 3!')

# Đóng tunnel cũ nếu có
ngrok.kill()

conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(8000, 'http')
PUBLIC_URL = tunnel.public_url

print('=' * 60)
print('  CTF Solver Backend READY')
print('=' * 60)
print(f'  URL   : {PUBLIC_URL}')
print(f'  Model : {MODEL_NAME}')
print()
print('  Copy và chạy trên máy local:')
print()
print(f'  export ANTHROPIC_BASE_URL={PUBLIC_URL}')
print( '  export ANTHROPIC_API_KEY=dummy')
print(f'  ctf --challenge ~/ctf/chall --model {MODEL_NAME}')
print('=' * 60)

In [ ]:
# Cell 6 — Test nhanh
import json, urllib.request

payload = json.dumps({
    'model': MODEL_NAME,
    'messages': [{'role': 'user', 'content': 'Reply with exactly: READY'}],
    'max_tokens': 10
}).encode()

req = urllib.request.Request(
    'http://localhost:8000/v1/chat/completions',
    data=payload,
    headers={'Content-Type': 'application/json'}
)

try:
    resp = json.loads(urllib.request.urlopen(req, timeout=30).read())
    print('✓ Test OK:', resp['choices'][0]['message']['content'])
except Exception as e:
    print(f'✗ Test failed: {e}')
    print('vLLM log tail:')
    !tail -20 /tmp/vllm.log

In [ ]:
# Cell 7 — Giữ session sống (để notebook mở, đừng đóng tab)
import time, urllib.request

print(f'Backend running: {PUBLIC_URL}')
print('Ctrl+C (Interrupt) để dừng.\n')

tick = 0
try:
    while True:
        time.sleep(30)
        tick += 1
        try:
            urllib.request.urlopen('http://localhost:8000/health', timeout=5)
            if tick % 10 == 0:  # Log mỗi 5 phút
                print(f'[{tick*30//60}m] vLLM alive')
        except Exception as e:
            print(f'WARNING: vLLM không phản hồi: {e}')
except KeyboardInterrupt:
    print('Stopped.')

## Debug — xem log vLLM
Chạy cell này nếu gặp lỗi ở bất kỳ bước nào.

In [ ]:
# Debug — 50 dòng log cuối
!tail -50 /tmp/vllm.log